# VOC Segmentation v3 (Full Training + Simple Baseline)

目标：
1) 使用 VOC2007 trainval 全量(422)训练；
2) 提供一个简单模型做快速自检，排查 mIoU 过低是否来自流程问题。

## 1) 超参数与路径

In [ ]:
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path('..').resolve()
VOC_ROOT = PROJECT_ROOT / 'DataSets' / 'VOC' / 'VOCtrainval_06-Nov-2007' / 'VOCdevkit' / 'VOC2007'
TRAINVAL_TXT = VOC_ROOT / 'ImageSets' / 'Segmentation' / 'trainval.txt'
IMAGE_DIR = VOC_ROOT / 'JPEGImages'
MASK_DIR = VOC_ROOT / 'SegmentationClass'

IMG_SIZE = (320, 320)
NUM_CLASSES = 21
IGNORE_INDEX = 255
BATCH_SIZE = 8
NUM_EPOCHS = 80
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
AMP_ENABLED = torch.cuda.is_available()

# simple baseline 自检参数
SANITY_SAMPLES = 16
SANITY_EPOCHS = 20

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
print(f'VOC_ROOT: {VOC_ROOT}')

## 2) 加载数据集

In [ ]:
def read_ids(path):
    return [x.strip() for x in Path(path).read_text().splitlines() if x.strip()]

all_ids = read_ids(TRAINVAL_TXT)
print(f'Trainval ids: {len(all_ids)}')

def dataset_sanity_check(ids):
    miss_img, miss_mask = 0, 0
    cls_hist = np.zeros(NUM_CLASSES, dtype=np.float64)
    total_valid = 0
    total_fg = 0
    for sid in ids:
        img_path = IMAGE_DIR / f'{sid}.jpg'
        mask_path = MASK_DIR / f'{sid}.png'
        if not img_path.exists():
            miss_img += 1
        if not mask_path.exists():
            miss_mask += 1
            continue
        m = np.array(Image.open(mask_path), dtype=np.int64)
        valid = m != IGNORE_INDEX
        vals = m[valid]
        if vals.size > 0:
            cls_hist += np.bincount(vals, minlength=NUM_CLASSES)
            total_valid += vals.size
            total_fg += (vals != 0).sum()
    fg_ratio = total_fg / max(total_valid, 1)
    print(f'missing image: {miss_img}, missing mask: {miss_mask}')
    print(f'foreground pixel ratio (exclude 255): {fg_ratio:.4f}')
    print('class pixel counts:', cls_hist.astype(np.int64))

dataset_sanity_check(all_ids)

class VOCSegDataset(Dataset):
    def __init__(self, ids, image_dir, mask_dir, img_size=(320, 320), train=False):
        self.ids = ids
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.img_size = img_size
        self.train = train

    def __len__(self):
        return len(self.ids)

    def _augment(self, image, mask):
        if random.random() < 0.5:
            image = TF.hflip(image)
            mask = TF.hflip(mask)
        if random.random() < 0.5:
            image = TF.vflip(image)
            mask = TF.vflip(mask)
        w, h = image.size
        crop_w = int(w * random.uniform(0.7, 1.0))
        crop_h = int(h * random.uniform(0.7, 1.0))
        crop_w = max(128, min(crop_w, w))
        crop_h = max(128, min(crop_h, h))
        top = random.randint(0, h - crop_h)
        left = random.randint(0, w - crop_w)
        image = TF.crop(image, top, left, crop_h, crop_w)
        mask = TF.crop(mask, top, left, crop_h, crop_w)
        return image, mask

    def __getitem__(self, idx):
        sid = self.ids[idx]
        image = Image.open(self.image_dir / f'{sid}.jpg').convert('RGB')
        mask = Image.open(self.mask_dir / f'{sid}.png')
        if self.train:
            image, mask = self._augment(image, mask)
        image = TF.resize(image, self.img_size, interpolation=InterpolationMode.BILINEAR)
        mask = TF.resize(mask, self.img_size, interpolation=InterpolationMode.NEAREST)
        image = TF.to_tensor(image)
        image = TF.normalize(image, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        mask = torch.from_numpy(np.array(mask, dtype=np.int64))
        return image, mask

train_dataset = VOCSegDataset(all_ids, IMAGE_DIR, MASK_DIR, IMG_SIZE, train=True)
train_eval_dataset = VOCSegDataset(all_ids, IMAGE_DIR, MASK_DIR, IMG_SIZE, train=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
train_eval_loader = DataLoader(train_eval_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'Full training size: {len(train_dataset)}')

## 3) 定义模型

In [ ]:
class TinySegNet(nn.Module):
    # 一个非常简单的编码器-解码器，用于快速验证流程
    def __init__(self, num_classes=21):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(inplace=True)
        )
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 2, stride=2), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, 2, stride=2), nn.ReLU(inplace=True),
            nn.Conv2d(32, num_classes, 1)
        )

    def forward(self, x):
        return self.dec(self.enc(x))

class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, p_drop=0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(p_drop) if p_drop > 0 else nn.Identity()
        )

    def forward(self, x):
        return self.block(x)

class UNetLite(nn.Module):
    def __init__(self, in_channels=3, num_classes=21, base=32):
        super().__init__()
        self.enc1 = ConvBNReLU(in_channels, base, p_drop=0.0)
        self.enc2 = ConvBNReLU(base, base * 2, p_drop=0.05)
        self.enc3 = ConvBNReLU(base * 2, base * 4, p_drop=0.1)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ConvBNReLU(base * 4, base * 8, p_drop=0.1)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, 2)
        self.dec3 = ConvBNReLU(base * 8, base * 4, p_drop=0.1)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.dec2 = ConvBNReLU(base * 4, base * 2, p_drop=0.05)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.dec1 = ConvBNReLU(base * 2, base, p_drop=0.0)
        self.cls = nn.Conv2d(base, num_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.cls(d1)

def multiclass_dice_loss(logits, target, num_classes=21, ignore_index=255, eps=1e-6):
    probs = torch.softmax(logits, dim=1)
    valid = target != ignore_index
    target = torch.where(valid, target, torch.zeros_like(target))
    one_hot = F.one_hot(target, num_classes=num_classes).permute(0, 3, 1, 2).float()
    valid = valid.unsqueeze(1)
    probs = probs * valid
    one_hot = one_hot * valid
    inter = (probs * one_hot).sum(dim=(0, 2, 3))
    den = probs.sum(dim=(0, 2, 3)) + one_hot.sum(dim=(0, 2, 3))
    dice = (2 * inter + eps) / (den + eps)
    return 1 - dice.mean()

def build_class_weights(ids, mask_dir, num_classes=21, ignore_index=255):
    hist = np.zeros(num_classes, dtype=np.float64)
    for sid in ids:
        m = np.array(Image.open(Path(mask_dir) / f'{sid}.png'), dtype=np.int64)
        valid = m != ignore_index
        vals = m[valid]
        hist += np.bincount(vals, minlength=num_classes)
    freq = hist / np.maximum(hist.sum(), 1.0)
    med = np.median(freq[freq > 0])
    w = med / np.maximum(freq, 1e-8)
    w = np.clip(w, 0.2, 8.0)
    return torch.tensor(w, dtype=torch.float32)

class CEDiceLoss(nn.Module):
    def __init__(self, class_weights, ce_weight=1.0, dice_weight=1.0, ignore_index=255, num_classes=21):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(ignore_index=ignore_index, weight=class_weights)
        self.ce_weight = ce_weight
        self.dice_weight = dice_weight
        self.ignore_index = ignore_index
        self.num_classes = num_classes

    def forward(self, logits, target):
        ce = self.ce(logits, target)
        dice = multiclass_dice_loss(logits, target, self.num_classes, self.ignore_index)
        return self.ce_weight * ce + self.dice_weight * dice

## 4) 初始化模型和优化器

In [ ]:
class_weights = build_class_weights(all_ids, MASK_DIR, num_classes=NUM_CLASSES, ignore_index=IGNORE_INDEX).to(DEVICE)
print('Class weights:', np.round(class_weights.detach().cpu().numpy(), 3))

# MODEL_NAME 可选："tiny" 或 "unet"
MODEL_NAME = 'unet'

if MODEL_NAME == 'tiny':
    model = TinySegNet(num_classes=NUM_CLASSES).to(DEVICE)
else:
    model = UNetLite(in_channels=3, num_classes=NUM_CLASSES, base=32).to(DEVICE)

criterion = CEDiceLoss(class_weights=class_weights, ce_weight=1.0, dice_weight=1.0, ignore_index=IGNORE_INDEX, num_classes=NUM_CLASSES)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: {MODEL_NAME}, Trainable params: {num_params / 1e6:.2f}M')

## 5) 训练

In [ ]:
@torch.no_grad()
def compute_metrics(model, loader, num_classes, ignore_index, device):
    model.eval()
    conf_mat = torch.zeros((num_classes, num_classes), dtype=torch.float64, device=device)
    for images, masks in loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        preds = model(images).argmax(dim=1)
        valid = masks != ignore_index
        gt = masks[valid]
        pd = preds[valid]
        k = gt * num_classes + pd
        conf_mat += torch.bincount(k, minlength=num_classes * num_classes).reshape(num_classes, num_classes)
    tp = conf_mat.diag()
    denom = conf_mat.sum(1) + conf_mat.sum(0) - tp
    iou = tp / torch.clamp(denom, min=1.0)
    pa = (tp.sum() / torch.clamp(conf_mat.sum(), min=1.0)).item()
    miou = iou.mean().item()
    fg_miou = iou[1:].mean().item()
    return pa, miou, fg_miou

def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    running_loss = 0.0
    for images, masks in loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, enabled=AMP_ENABLED):
            logits = model(images)
            loss = criterion(logits, masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
    return running_loss / max(len(loader), 1)

# --- quick sanity: 用 tiny 模型在少量样本上过拟合，验证流程是否通 ---
tiny_model = TinySegNet(num_classes=NUM_CLASSES).to(DEVICE)
tiny_opt = torch.optim.Adam(tiny_model.parameters(), lr=1e-3)
tiny_criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
sanity_ids = all_ids[:SANITY_SAMPLES]
sanity_ds = VOCSegDataset(sanity_ids, IMAGE_DIR, MASK_DIR, IMG_SIZE, train=False)
sanity_loader = DataLoader(sanity_ds, batch_size=4, shuffle=True, num_workers=0)

for ep in range(1, SANITY_EPOCHS + 1):
    tiny_model.train()
    ep_loss = 0.0
    for images, masks in sanity_loader:
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)
        tiny_opt.zero_grad()
        logits = tiny_model(images)
        loss = tiny_criterion(logits, masks)
        loss.backward()
        tiny_opt.step()
        ep_loss += loss.item()
    if ep % 5 == 0 or ep == 1:
        pa, miou, fg_miou = compute_metrics(tiny_model, sanity_loader, NUM_CLASSES, IGNORE_INDEX, DEVICE)
        print(f'[Sanity Tiny] Epoch {ep:02d}/{SANITY_EPOCHS} Loss:{ep_loss/len(sanity_loader):.4f} PA:{pa:.4f} mIoU:{miou:.4f} fg-mIoU:{fg_miou:.4f}')

# --- full training on 422 samples ---
best_fg_miou = -1.0
best_model_path = Path('best_unet_voc_v3_full.pth')

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, scaler, DEVICE)
    pa, miou, fg_miou = compute_metrics(model, train_eval_loader, NUM_CLASSES, IGNORE_INDEX, DEVICE)
    scheduler.step()
    if fg_miou > best_fg_miou:
        best_fg_miou = fg_miou
        torch.save(model.state_dict(), best_model_path)
    lr = optimizer.param_groups[0]['lr']
    print(f'Epoch [{epoch:02d}/{NUM_EPOCHS}] LR:{lr:.2e} TrainLoss:{train_loss:.4f} TrainPA:{pa:.4f} TrainmIoU:{miou:.4f} Trainfg-mIoU:{fg_miou:.4f}')

print(f'Best Train fg-mIoU: {best_fg_miou:.4f}')
print(f'Best model saved to: {best_model_path.resolve()}')

## 6) 评估

In [ ]:
model.load_state_dict(torch.load('best_unet_voc_v3_full.pth', map_location=DEVICE))
pa, miou, fg_miou = compute_metrics(model, train_eval_loader, NUM_CLASSES, IGNORE_INDEX, DEVICE)
print(f'[Full-Train Eval] Pixel Accuracy: {pa:.4f}')
print(f'[Full-Train Eval] mIoU: {miou:.4f}')
print(f'[Full-Train Eval] Foreground mIoU: {fg_miou:.4f}')

model.eval()
images, masks = next(iter(train_eval_loader))
images = images.to(DEVICE)
with torch.no_grad():
    preds = model(images).argmax(dim=1).cpu()

show_n = min(3, images.size(0))
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
fig, axes = plt.subplots(show_n, 3, figsize=(10, 4 * show_n))
if show_n == 1:
    axes = np.expand_dims(axes, axis=0)
for i in range(show_n):
    img = images[i].cpu() * std + mean
    img = img.permute(1, 2, 0).clamp(0, 1).numpy()
    axes[i, 0].imshow(img)
    axes[i, 0].set_title('Image')
    axes[i, 0].axis('off')
    axes[i, 1].imshow(masks[i].numpy(), cmap='tab20', vmin=0, vmax=NUM_CLASSES - 1)
    axes[i, 1].set_title('GT Mask')
    axes[i, 1].axis('off')
    axes[i, 2].imshow(preds[i].numpy(), cmap='tab20', vmin=0, vmax=NUM_CLASSES - 1)
    axes[i, 2].set_title('Pred Mask')
    axes[i, 2].axis('off')
plt.tight_layout()
plt.show()